<a href="https://colab.research.google.com/github/marcusslongweek/Homework/blob/main/CS4410HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 Homework — Exercises 16.1 and 16.4



In [ ]:
import time
import pandas as pd
from tensorflow.keras.datasets import mnist, fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, MaxPooling2D
from tensorflow.keras.utils import to_categorical

## Helper Functions

The preprocessing follows the chapter example: reshape to `(28, 28, 1)`, scale pixel values to 0–1, and one-hot encode the labels.

In [ ]:
def prepare_data(dataset):
    (X_train, y_train), (X_test, y_test) = dataset.load_data()

    X_train = X_train.reshape((60000, 28, 28, 1))
    X_test = X_test.reshape((10000, 28, 28, 1))

    X_train = X_train.astype('float32') / 255
    X_test = X_test.astype('float32') / 255

    y_train = to_categorical(y_train)
    y_test = to_categorical(y_test)

    return X_train, y_train, X_test, y_test


def compile_model(cnn):
    cnn.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return cnn


def build_original_cnn():
    cnn = Sequential()
    cnn.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu',
                   input_shape=(28, 28, 1)))
    cnn.add(MaxPooling2D(pool_size=(2, 2)))
    cnn.add(Conv2D(filters=128, kernel_size=(3, 3), activation='relu'))
    cnn.add(MaxPooling2D(pool_size=(2, 2)))
    cnn.add(Flatten())
    cnn.add(Dense(units=128, activation='relu'))
    cnn.add(Dense(units=10, activation='softmax'))
    return compile_model(cnn)


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    start_time = time.time()

    history = model.fit(
        X_train,
        y_train,
        epochs=5,
        batch_size=64,
        validation_split=0.1
    )

    training_time = time.time() - start_time
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

    return history, loss, accuracy, training_time

## Part 1 — Baseline: Original CNN on MNIST

In [ ]:
X_train_mnist, y_train_mnist, X_test_mnist, y_test_mnist = prepare_data(mnist)

mnist_cnn = build_original_cnn()
mnist_cnn.summary()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       409,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 485,514 (1.85 MB)

 Trainable params: 485,514 (1.85 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
mnist_history, mnist_loss, mnist_accuracy, mnist_time = train_and_evaluate(
    mnist_cnn,
    X_train_mnist,
    y_train_mnist,
    X_test_mnist,
    y_test_mnist
)

print(f"MNIST test loss: {mnist_loss:.4f}")
print(f"MNIST test accuracy: {mnist_accuracy:.4%}")
print(f"MNIST training time: {mnist_time:.2f} seconds")

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 123s 144ms/step - accuracy: 0.9549 - loss: 0.1474 - val_accuracy: 0.9867 - val_loss: 0.0479
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 116s 137ms/step - accuracy: 0.9863 - loss: 0.0444 - val_accuracy: 0.9898 - val_loss: 0.0363
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 114s 135ms/step - accuracy: 0.9907 - loss: 0.0290 - val_accuracy: 0.9912 - val_loss: 0.0312
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 136s 162ms/step - accuracy: 0.9930 - loss: 0.0221 - val_accuracy: 0.9907 - val_loss: 0.0336
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 115s 136ms/step - accuracy: 0.9947 - loss: 0.0163 - val_accuracy: 0.9910 - val_loss: 0.0344
MNIST test loss: 0.0273
MNIST test accuracy: 99.2200%
MNIST training time: 604.74 seconds


## Part 2 — Exercise 16.1: Fashion-MNIST

Use the same CNN architecture and training settings so the comparison with MNIST is fair.

In [ ]:
X_train_fashion, y_train_fashion, X_test_fashion, y_test_fashion = prepare_data(fashion_mnist)

fashion_cnn = build_original_cnn()
fashion_cnn.summary()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 13, 13, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       409,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 485,514 (1.85 MB)

 Trainable params: 485,514 (1.85 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
fashion_history, fashion_loss, fashion_accuracy, fashion_time = train_and_evaluate(
    fashion_cnn,
    X_train_fashion,
    y_train_fashion,
    X_test_fashion,
    y_test_fashion
)

print(f"Fashion-MNIST test loss: {fashion_loss:.4f}")
print(f"Fashion-MNIST test accuracy: {fashion_accuracy:.4%}")
print(f"Fashion-MNIST training time: {fashion_time:.2f} seconds")

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 136s 158ms/step - accuracy: 0.8336 - loss: 0.4643 - val_accuracy: 0.8660 - val_loss: 0.3605
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 121s 143ms/step - accuracy: 0.8883 - loss: 0.3061 - val_accuracy: 0.8930 - val_loss: 0.2969
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 138s 139ms/step - accuracy: 0.9046 - loss: 0.2583 - val_accuracy: 0.9053 - val_loss: 0.2677
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 118s 139ms/step - accuracy: 0.9172 - loss: 0.2248 - val_accuracy: 0.9115 - val_loss: 0.2605
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 159s 159ms/step - accuracy: 0.9264 - loss: 0.1991 - val_accuracy: 0.9102 - val_loss: 0.2484
Fashion-MNIST test loss: 0.2607
Fashion-MNIST test accuracy: 90.5600%
Fashion-MNIST training time: 678.96 seconds


In [ ]:
comparison_16_1 = pd.DataFrame({
    "Dataset": ["MNIST", "Fashion-MNIST"],
    "Test Accuracy": [mnist_accuracy, fashion_accuracy],
    "Training Time (seconds)": [mnist_time, fashion_time]
})
comparison_16_1

,Dataset,Test Accuracy,Training Time (seconds)
0,MNIST,0.9922,604.741537
1,Fashion-MNIST,0.9056,678.964041


### Exercise 16.1 — Written Comparison

**How well does the model perform on Fashion-MNIST compared with MNIST?**  
The Fashion-MNIST model was 12.27% slower and 8.72% less accurate when compared to MNIST

# Exercise 16.4 — Convnet Layers

## Part 3 — Remove the First Dense Layer

In [ ]:
def build_cnn_without_dense_128():
    cnn = Sequential()
    cnn.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu',
                   input_shape=(28, 28, 1)))
    cnn.add(MaxPooling2D(pool_size=(2, 2)))
    cnn.add(Conv2D(filters=128, kernel_size=(3, 3), activation='relu'))
    cnn.add(MaxPooling2D(pool_size=(2, 2)))
    cnn.add(Flatten())

    # Dense(128) removed
    cnn.add(Dense(units=10, activation='softmax'))

    return compile_model(cnn)


no_dense_cnn = build_cnn_without_dense_128()
no_dense_cnn.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 26, 26, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 13, 13, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 11, 11, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 3200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │        32,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,506 (416.04 KB)

 Trainable params: 106,506 (416.04 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
no_dense_history, no_dense_loss, no_dense_accuracy, no_dense_time = train_and_evaluate(
    no_dense_cnn,
    X_train_mnist,
    y_train_mnist,
    X_test_mnist,
    y_test_mnist
)

print(f"No-Dense-128 test loss: {no_dense_loss:.4f}")
print(f"No-Dense-128 test accuracy: {no_dense_accuracy:.4%}")
print(f"No-Dense-128 training time: {no_dense_time:.2f} seconds")

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 115s 135ms/step - accuracy: 0.9492 - loss: 0.1727 - val_accuracy: 0.9863 - val_loss: 0.0475
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 150s 145ms/step - accuracy: 0.9842 - loss: 0.0513 - val_accuracy: 0.9852 - val_loss: 0.0488
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 139s 142ms/step - accuracy: 0.9889 - loss: 0.0369 - val_accuracy: 0.9877 - val_loss: 0.0455
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 113s 134ms/step - accuracy: 0.9910 - loss: 0.0283 - val_accuracy: 0.9910 - val_loss: 0.0351
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 113s 134ms/step - accuracy: 0.9932 - loss: 0.0217 - val_accuracy: 0.9915 - val_loss: 0.0300
No-Dense-128 test loss: 0.0276
No-Dense-128 test accuracy: 99.1000%
No-Dense-128 training time: 630.93 seconds


## Part 4 — Add a Dense Layer with 4096 Neurons

In [ ]:
def build_cnn_with_dense_4096():
    cnn = Sequential()
    cnn.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu',
                   input_shape=(28, 28, 1)))
    cnn.add(MaxPooling2D(pool_size=(2, 2)))
    cnn.add(Conv2D(filters=128, kernel_size=(3, 3), activation='relu'))
    cnn.add(MaxPooling2D(pool_size=(2, 2)))
    cnn.add(Flatten())

    # Added before the two original Dense layers
    cnn.add(Dense(units=4096, activation='relu'))
    cnn.add(Dense(units=128, activation='relu'))
    cnn.add(Dense(units=10, activation='softmax'))

    return compile_model(cnn)


dense_4096_cnn = build_cnn_with_dense_4096()
dense_4096_cnn.summary()

In [ ]:
dense_4096_history, dense_4096_loss, dense_4096_accuracy, dense_4096_time = train_and_evaluate(
    dense_4096_cnn,
    X_train_mnist,
    y_train_mnist,
    X_test_mnist,
    y_test_mnist
)

print(f"Dense-4096 test loss: {dense_4096_loss:.4f}")
print(f"Dense-4096 test accuracy: {dense_4096_accuracy:.4%}")
print(f"Dense-4096 training time: {dense_4096_time:.2f} seconds")

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 389s 459ms/step - accuracy: 0.9613 - loss: 0.1231 - val_accuracy: 0.9830 - val_loss: 0.0529
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 457s 476ms/step - accuracy: 0.9882 - loss: 0.0380 - val_accuracy: 0.9887 - val_loss: 0.0365
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 397s 470ms/step - accuracy: 0.9916 - loss: 0.0259 - val_accuracy: 0.9880 - val_loss: 0.0416
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 397s 471ms/step - accuracy: 0.9944 - loss: 0.0180 - val_accuracy: 0.9910 - val_loss: 0.0357
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 385s 457ms/step - accuracy: 0.9950 - loss: 0.0153 - val_accuracy: 0.9912 - val_loss: 0.0377
Dense-4096 test loss: 0.0326
Dense-4096 test accuracy: 99.1200%
Dense-4096 training time: 2025.44 seconds


In [ ]:
comparison_16_4 = pd.DataFrame({
    "Model": ["Original CNN", "Without Dense(128)", "With Dense(4096)"],
    "Test Accuracy": [mnist_accuracy, no_dense_accuracy, dense_4096_accuracy],
    "Training Time (seconds)": [mnist_time, no_dense_time, dense_4096_time]
})
comparison_16_4

,Model,Test Accuracy,Training Time (seconds)
0,Original CNN,0.9922,604.741537
1,Without Dense(128),0.9899,671.778706
2,With Dense(4096),0.9912,2025.441962


### Exercise 16.4 — Written Comparison

**What happens when the first Dense layer is removed?**  
The accuracy decreased by about 0.23% and the time increased by about 11%.

**What happens when a Dense layer with 4096 neurons is added before the two original Dense layers?**  
The Accuracy decreased by 0.10%, but the time more than tripled (334.92% increase).